In [2]:
# Installation des dépendances
!pip install scenedetect ultralytics open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.9/130.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.


In [3]:
import os, cv2, torch, numpy as np, pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
import open_clip
from PIL import Image
from collections import defaultdict

# Configuration GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_model = YOLO('yolov8n.pt').to(device)
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model = clip_model.to(device)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [4]:
def process_semantic_from_keyframes(video_id, image_paths):
    frames_for_clip = []
    yolo_counts = {'person': 0, 'skis': 0, 'snowboard': 0}

    # On trie pour garder l'ordre kf0, kf1...
    image_paths.sort()

    for img_path in image_paths:
        frame = cv2.imread(img_path)
        if frame is None: continue

        # YOLO (Detection d'objets)
        y_res = yolo_model(frame, imgsz=320, verbose=False, conf=0.25)[0]
        for c in y_res.boxes.cls:
            label = y_res.names[int(c)]
            if label in yolo_counts: yolo_counts[label] += 1

        # Préparation CLIP
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames_for_clip.append(preprocess(Image.fromarray(frame_rgb)).unsqueeze(0))

    # CLIP BATCH (Extraction d'ambiance)
    avg_embedding = np.zeros(512)
    if frames_for_clip:
        img_input = torch.cat(frames_for_clip).to(device)
        with torch.no_grad():
            # On encode toutes les frames d'un coup et on fait la moyenne
            emb = clip_model.encode_image(img_input).mean(dim=0).cpu().numpy()
            avg_embedding = emb

    # Formatage des résultats (ID + YOLO + 512 colonnes CLIP)
    res = {
        'video_id': video_id,
        'yolo_person': yolo_counts['person'],
        'yolo_ski_snow': yolo_counts['skis'] + yolo_counts['snowboard']
    }
    for i, v in enumerate(avg_embedding):
        res[f'c{i}'] = round(float(v), 4)

    return res

In [9]:
from google.colab import drive

# 1. Montage du Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
KEYFRAMES_DIR = "/content/drive/MyDrive/hackathon/keyframes/"

# 1. Grouper les images par vidéo
video_groups = defaultdict(list)
for f in os.listdir(KEYFRAMES_DIR):
    if f.lower().endswith(('.jpg', '.jpeg')):
        video_id = f.rsplit('_kf', 1)[0]
        video_groups[video_id].append(os.path.join(KEYFRAMES_DIR, f))

# 2. Lancer l'analyse
results = []
print(f"🚀 Analyse sémantique de {len(video_groups)} vidéos via keyframes...")

for v_id, img_paths in tqdm(video_groups.items()):

  data = process_semantic_from_keyframes(v_id, img_paths)
  if data:
     results.append(data)

# 3. Sauvegarde finale
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/hackathon/features_semantics.csv", index=False)

🚀 Analyse sémantique de 1686 vidéos via keyframes...


100%|██████████| 1686/1686 [1:51:19<00:00,  3.96s/it]
